# Edgelist Network Analysis and Epidemic Modeling

## Introduction

This notebook demonstrates how to work with **real-world network data** loaded from edgelist files in the epidemic modeling workflow.

We will:

- **Load a network from an edgelist file**
- **Integrate it with the existing workflow functions**
- **Analyze network properties** (degree distribution, centrality measures, etc.)
- **Run epidemic simulations** on the real network data
- **Compare with synthetic network types**

This approach allows you to study epidemic dynamics on empirical network structures, such as social networks, contact networks, or transportation networks.

## Setup

First, we'll set up the environment and create necessary directories for storing results.

In [47]:
# Navigate to the directory containing this file
cd(@__DIR__)

# Create directories if they don't exist
if !isdir("data")
    mkdir("data")
end
if !isdir("figures")
    mkdir("figures")
end
if !isdir("output")
    mkdir("output")
end

Next, we activate the current directory as the environment for Pkg and import the required packages. This includes the additional `GraphIO.EdgeList` package for loading edgelist files.

In [48]:
# Import the Pkg module, activate the current directory as the environment for Pkg, instantiate the environment
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

# Import the necessary packages
using Agents, Graphs, Random, Plots, DataFrames, CSV, CategoricalArrays, Statistics, StatsBase, StatsPlots, Distributions, Measures, Optim
using GraphIO.EdgeList, GraphPlot, MetaGraphs, JSON

  Activating project at `c:\Users\Leonard\julia-workspace\espidam-tutorial-2025`


## Loading Network from Edgelist

We'll load a network from an edgelist file. The edgelist format is a common way to represent networks where each line contains two node IDs representing an edge.

The code below:
1. Loads the graph from the edgelist file
2. Removes any duplicate edges
3. Converts it to an undirected simple graph
4. Visualizes the network structure

In [49]:
# Read file
edge_lines = readlines("degs/fullnetwork_2000_edgelist.txt")

# Transform each line
transformed_edges = [replace(line, " -> " => ",") for line in edge_lines]

# Write to a new file
write("degs/network_n2000", join(transformed_edges, "\n"))

# Load graph from edgelist file
graph = loadgraph("degs/network_n2000", "graph_key", EdgeListFormat())

# Convert to undirected simple graph, removes all double edges
graph = SimpleGraph(graph)

println("Loaded network with $(nv(graph)) nodes and $(ne(graph)) edges")
println("Mean degree: $(round(2 * ne(graph) / nv(graph), digits=2))")

Loaded network with 1860 nodes and 9308 edges
Mean degree: 10.01


In [50]:
# Visualize the network (if it's not too large)
if nv(graph) <= 200  # Only plot if network is reasonably small
    gplot(graph)
else
    println("Network too large for visualization ($(nv(graph)) nodes)")
    println("Consider using a subset or different visualization approach")
end

Network too large for visualization (1860 nodes)
Consider using a subset or different visualization approach


## Adding Node Metadata with MetaGraphs

Now we'll load the node attributes from the JSON file and create a MetaGraph to store both the network structure and node metadata. This allows us to associate properties like age, gender, and behavioral characteristics with each node in the network.

In [51]:
# Load node attributes from JSON file
attributes_data = JSON.parsefile("degs/n2000_attributes.json")
node_attributes = attributes_data["nodes"]

println("Loaded $(length(node_attributes)) node attributes")
println("Available attributes: $(join(keys(node_attributes[1]), ", "))")

# Display sample of the data
println("\nSample node attributes:")
for i in 1:min(5, length(node_attributes))
    node = node_attributes[i]
    println("Node $(node["Name"]): Age=$(node["Age"]), Gender=$(node["Gender"]), PercSevHigh=$(node["percievedSeverityHigh"])")
end

Loaded 2000 node attributes
Available attributes: percievedSeverityHigh, protectiveMeasureScore, contactRestricting, Age, Gender, Name

Sample node attributes:
Node 1: Age=76.0, Gender=MALE, PercSevHigh=true
Node 2: Age=62.0, Gender=MALE, PercSevHigh=false
Node 3: Age=79.0, Gender=MALE, PercSevHigh=false
Node 4: Age=66.0, Gender=MALE, PercSevHigh=false
Node 5: Age=44.0, Gender=FEMALE, PercSevHigh=true


In [52]:
# Create a MetaGraph from the simple graph
meta_graph = MetaGraph(graph)

println("Created MetaGraph with $(nv(meta_graph)) nodes and $(ne(meta_graph)) edges")

# Add node attributes to the MetaGraph
# Note: JSON node names are strings, but graph vertices are integers
for node_data in node_attributes
    node_id = parse(Int, node_data["Name"])
    
    # Only add attributes if the node exists in the graph
    if node_id <= nv(meta_graph)
        # Parse and set node properties
        set_prop!(meta_graph, node_id, :age, parse(Float64, node_data["Age"]))
        set_prop!(meta_graph, node_id, :gender, node_data["Gender"])
        set_prop!(meta_graph, node_id, :perceived_severity_high, parse(Bool, node_data["percievedSeverityHigh"]))
        set_prop!(meta_graph, node_id, :contact_restricting, parse(Bool, node_data["contactRestricting"]))
        set_prop!(meta_graph, node_id, :protective_measure_score, parse(Float64, node_data["protectiveMeasureScore"]))
    end
end

println("Added node attributes to MetaGraph")

# Verify the attributes were added correctly
sample_node = 1
if has_prop(meta_graph, sample_node, :age)
    println("\nSample node $sample_node attributes:")
    println("  Age: $(get_prop(meta_graph, sample_node, :age))")
    println("  Gender: $(get_prop(meta_graph, sample_node, :gender))")
    println("  Perceived Severity High: $(get_prop(meta_graph, sample_node, :perceived_severity_high))")
    println("  Contact Restricting: $(get_prop(meta_graph, sample_node, :contact_restricting))")
    println("  Protective Measure Score: $(get_prop(meta_graph, sample_node, :protective_measure_score))")
else
    println("Warning: No attributes found for node $sample_node")
end

Created MetaGraph with 1860 nodes and 9308 edges
Added node attributes to MetaGraph

Sample node 1 attributes:
  Age: 76.0
  Gender: MALE
  Perceived Severity High: true
  Contact Restricting: true
  Protective Measure Score: 2.0


In [53]:
# Analyze the node attributes distribution
println("=== Node Attributes Analysis ===")

# Age distribution
ages = [get_prop(meta_graph, i, :age) for i in 1:nv(meta_graph) if has_prop(meta_graph, i, :age)]
println("Age statistics:")
println("  Mean age: $(round(mean(ages), digits=1))")
println("  Age range: $(round(minimum(ages), digits=1)) - $(round(maximum(ages), digits=1))")
println("  Standard deviation: $(round(std(ages), digits=1))")

# Gender distribution
genders = [get_prop(meta_graph, i, :gender) for i in 1:nv(meta_graph) if has_prop(meta_graph, i, :gender)]
gender_counts = countmap(genders)
println("\nGender distribution:")
for (gender, count) in gender_counts
    println("  $gender: $count ($(round(100*count/length(genders), digits=1))%)")
end

# Perceived severity distribution
perc_sev = [get_prop(meta_graph, i, :perceived_severity_high) for i in 1:nv(meta_graph) if has_prop(meta_graph, i, :perceived_severity_high)]
high_sev_count = sum(perc_sev)
println("\nPerceived severity high:")
println("  True: $high_sev_count ($(round(100*high_sev_count/length(perc_sev), digits=1))%)")
println("  False: $(length(perc_sev) - high_sev_count) ($(round(100*(length(perc_sev) - high_sev_count)/length(perc_sev), digits=1))%)")

# Contact restricting behavior
contact_rest = [get_prop(meta_graph, i, :contact_restricting) for i in 1:nv(meta_graph) if has_prop(meta_graph, i, :contact_restricting)]
contact_rest_count = sum(contact_rest)
println("\nContact restricting behavior:")
println("  True: $contact_rest_count ($(round(100*contact_rest_count/length(contact_rest), digits=1))%)")
println("  False: $(length(contact_rest) - contact_rest_count) ($(round(100*(length(contact_rest) - contact_rest_count)/length(contact_rest), digits=1))%)")

# Protective measure scores
prot_scores = [get_prop(meta_graph, i, :protective_measure_score) for i in 1:nv(meta_graph) if has_prop(meta_graph, i, :protective_measure_score)]
println("\nProtective measure score statistics:")
println("  Mean: $(round(mean(prot_scores), digits=2))")
println("  Range: $(minimum(prot_scores)) - $(maximum(prot_scores))")
println("  Standard deviation: $(round(std(prot_scores), digits=2))")

score_counts = countmap(prot_scores)
println("  Distribution:")
for score in sort(collect(keys(score_counts)))
    count = score_counts[score]
    println("    Score $score: $count ($(round(100*count/length(prot_scores), digits=1))%)")
end

=== Node Attributes Analysis ===
Age statistics:
  Mean age: 67.5
  Age range: 25.0 - 97.0
  Standard deviation: 9.4

Gender distribution:
  FEMALE: 695 (40.3%)
  MALE: 1031 (59.7%)
Age statistics:
  Mean age: 67.5
  Age range: 25.0 - 97.0
  Standard deviation: 9.4

Gender distribution:
  FEMALE: 695 (40.3%)
  MALE: 1031 (59.7%)

Perceived severity high:
  True: 741 (42.9%)
  False: 985 (57.1%)

Contact restricting behavior:
  True: 1269 (73.5%)
  False: 457 (26.5%)

Perceived severity high:
  True: 741 (42.9%)
  False: 985 (57.1%)

Contact restricting behavior:
  True: 1269 (73.5%)
  False: 457 (26.5%)

Protective measure score statistics:
  Mean: 2.22
  Range: 0.0 - 4.0
  Standard deviation: 1.25
  Distribution:
    Score 0.0: 174 (10.1%)
    Score 1.0: 337 (19.5%)
    Score 2.0: 491 (28.4%)
    Score 3.0: 375 (21.7%)
    Score 4.0: 349 (20.2%)

Protective measure score statistics:
  Mean: 2.22
  Range: 0.0 - 4.0
  Standard deviation: 1.25
  Distribution:
    Score 0.0: 174 (10.1%)
 

## Using MetaGraph with Node Attributes in Model Initialization

Now we can use the MetaGraph with node attributes in our epidemic model. This allows us to leverage the node metadata for more realistic simulations, such as using age to determine risk groups or using protective measure scores to influence transmission probabilities.

In [54]:
# Update the graph variable to use our MetaGraph with attributes
# This will be used in the subsequent model initialization
graph = meta_graph

println("Updated graph to use MetaGraph with $(nv(graph)) nodes and $(ne(graph)) edges")
println("Graph now includes node attributes from the JSON file")

# Demonstrate accessing attributes through the graph
println("\nExample: Accessing attributes for the first 3 nodes:")
for node_id in 1:min(3, nv(graph))
    if has_prop(graph, node_id, :age)
        age = get_prop(graph, node_id, :age)
        gender = get_prop(graph, node_id, :gender)
        prot_score = get_prop(graph, node_id, :protective_measure_score)
        println("  Node $node_id: Age=$age, Gender=$gender, ProtScore=$prot_score")
    end
end

Updated graph to use MetaGraph with 1860 nodes and 9308 edges
Graph now includes node attributes from the JSON file

Example: Accessing attributes for the first 3 nodes:
  Node 1: Age=76.0, Gender=MALE, ProtScore=2.0
  Node 2: Age=62.0, Gender=MALE, ProtScore=2.0
  Node 3: Age=79.0, Gender=MALE, ProtScore=4.0


In [61]:
# Create an enhanced agent initialization function that uses node attributes
function initialize_with_attributes(meta_graph; patient_zero=:random, trans_prob=0.1, days_to_recovered=14, seed=42)
    Random.seed!(seed)
    
    # Extract the underlying simple graph for ABM (MetaGraphs can't be used directly)
    simple_graph = SimpleGraph(meta_graph)
    n_nodes = nv(simple_graph)
    
    # Create the model properties 
    properties = Dict(
        :network_type => :edgelist_with_attributes,
        :n_nodes => n_nodes,
        :mean_degree => round(2 * ne(simple_graph) / n_nodes, digits=2),
        :susceptible_count => n_nodes - 1,  # Will be updated in model_step!
        :infected_count => 1,               # Starting with patient zero
        :hospitalized_count => 0,
        :recovered_count => 0,
        :trans_prob => trans_prob,
        :days_to_recovered => days_to_recovered,
        :meta_graph => meta_graph,  # Store MetaGraph for accessing node attributes
        :graph => simple_graph      # Store simple graph for compatibility
    )
    
    # Create the space and model using the simple graph
    space = GraphSpace(simple_graph)
    model = StandardABM(Person, space; agent_step!, model_step!, properties=properties)
    
    # Initialize agents with attributes-based risk assignment
    for node_id in 1:n_nodes
        # Determine risk based on age and perceived severity from attributes
        risk = :low  # default
        if has_prop(meta_graph, node_id, :age) && has_prop(meta_graph, node_id, :perceived_severity_high)
            age = get_prop(meta_graph, node_id, :age)
            perceived_sev_high = get_prop(meta_graph, node_id, :perceived_severity_high)
            
            # High risk if age > 65 OR perceived severity is high
            if age > 65.0 || perceived_sev_high
                risk = :high
            end
        end
        
        # Create and add agent
        agent = Person(node_id, node_id, :S, 0, risk)  # id, pos, status, days_infected, risk
        add_agent_own_pos!(agent, model)
    end
    
    # Set patient zero
    if patient_zero == :random
        patient_id = rand(1:n_nodes)
    elseif patient_zero == :highest_degree
        degrees = [degree(simple_graph, i) for i in 1:n_nodes]
        patient_id = argmax(degrees)
    elseif patient_zero isa Integer
        patient_id = patient_zero
    end
    
    model[patient_id].status = :I
    model[patient_id].days_infected = 1
    
    return model
end

println("Created enhanced initialization function that uses node attributes for risk assignment")

Created enhanced initialization function that uses node attributes for risk assignment


In [62]:
# Initialize the model using our enhanced function with node attributes
model_with_attributes = initialize_with_attributes(
    meta_graph,  # Pass the MetaGraph, function will extract simple graph for ABM
    patient_zero = :random,
    trans_prob = 0.1,
    days_to_recovered = 14,
    seed = 42
)

println("Model initialized with node attributes!")
println("Network type: $(model_with_attributes.network_type)")
println("Number of nodes: $(model_with_attributes.n_nodes)")
println("Mean degree: $(model_with_attributes.mean_degree)")

# Analyze risk distribution based on attributes
risk_counts = countmap([agent.risk for agent in allagents(model_with_attributes)])
println("\nRisk distribution based on node attributes:")
for (risk, count) in risk_counts
    percentage = round(100 * count / model_with_attributes.n_nodes, digits=1)
    println("  $risk risk: $count agents ($percentage%)")
end

# Show some example agents with their graph attributes
println("\nExample agents with their node attributes:")
for agent_id in 1:min(5, model_with_attributes.n_nodes)
    agent = model_with_attributes[agent_id]
    # Access attributes from the MetaGraph stored in model properties
    if has_prop(model_with_attributes.meta_graph, agent_id, :age)
        age = get_prop(model_with_attributes.meta_graph, agent_id, :age)
        gender = get_prop(model_with_attributes.meta_graph, agent_id, :gender)
        prot_score = get_prop(model_with_attributes.meta_graph, agent_id, :protective_measure_score)
        contact_rest = get_prop(model_with_attributes.meta_graph, agent_id, :contact_restricting)
        println("  Agent $agent_id: Risk=$(agent.risk), Age=$age, Gender=$gender, ProtScore=$prot_score, ContactRestrict=$contact_rest")
    end
end

Model initialized with node attributes!
Network type: edgelist_with_attributes
Number of nodes: 1860
Mean degree: 10.01

Risk distribution based on node attributes:
  high risk: 1313 agents (70.6%)
  low risk: 547 agents (29.4%)

Example agents with their node attributes:
  Agent 1: Risk=high, Age=76.0, Gender=MALE, ProtScore=2.0, ContactRestrict=true
  Agent 2: Risk=low, Age=62.0, Gender=MALE, ProtScore=2.0, ContactRestrict=true
  Agent 3: Risk=high, Age=79.0, Gender=MALE, ProtScore=4.0, ContactRestrict=true
  Agent 4: Risk=high, Age=66.0, Gender=MALE, ProtScore=1.0, ContactRestrict=true
  Agent 5: Risk=high, Age=44.0, Gender=FEMALE, ProtScore=0.0, ContactRestrict=true

  high risk: 1313 agents (70.6%)
  low risk: 547 agents (29.4%)

Example agents with their node attributes:
  Agent 1: Risk=high, Age=76.0, Gender=MALE, ProtScore=2.0, ContactRestrict=true
  Agent 2: Risk=low, Age=62.0, Gender=MALE, ProtScore=2.0, ContactRestrict=true
  Agent 3: Risk=high, Age=79.0, Gender=MALE, ProtSc

In [63]:
# Helper function to get node attributes from the model
function get_node_attribute(model, node_id, attribute)
    """Get a node attribute from the MetaGraph stored in the model"""
    if has_prop(model.meta_graph, node_id, attribute)
        return get_prop(model.meta_graph, node_id, attribute)
    else
        return nothing
    end
end

# Helper function to check if node has attribute
function has_node_attribute(model, node_id, attribute)
    """Check if a node has a specific attribute in the MetaGraph"""
    return has_prop(model.meta_graph, node_id, attribute)
end

println("Helper functions created for accessing node attributes from the model")

Helper functions created for accessing node attributes from the model


### Important Note: MetaGraphs and ABM Compatibility

**The Challenge**: MetaGraphs cannot be used directly as the graph space in Agents.jl ABM models. The `GraphSpace` constructor requires a simple graph structure.

**The Solution**: We use a two-graph approach:
1. **Simple Graph**: Used for the ABM space (required by Agents.jl)
2. **MetaGraph**: Stored as a model property to access node attributes

This approach is similar to what's done in `metagraphs_testing.jl` - the MetaGraph contains the metadata, but the ABM operates on the underlying simple graph structure. Agent interactions can then access node attributes through the MetaGraph when needed.

**Key Benefits**:
- ABM functionality works normally with the simple graph
- Node attributes remain accessible via the MetaGraph
- No loss of metadata information
- Compatible with existing ABM workflows

In [64]:
# Example: Using node attributes during simulation
println("=== Example: Accessing Node Attributes ===")

# Get attributes for the first few agents using helper functions
for agent_id in 1:min(3, model_with_attributes.n_nodes)
    agent = model_with_attributes[agent_id]
    
    if has_node_attribute(model_with_attributes, agent_id, :age)
        age = get_node_attribute(model_with_attributes, agent_id, :age)
        gender = get_node_attribute(model_with_attributes, agent_id, :gender)
        prot_score = get_node_attribute(model_with_attributes, agent_id, :protective_measure_score)
        
        println("Agent $agent_id ($(agent.status), $(agent.risk) risk):")
        println("  Age: $age, Gender: $gender, Protection Score: $prot_score")
    end
end

# You can also access the MetaGraph directly when needed
println("\nDirect MetaGraph access example:")
meta_graph_ref = model_with_attributes.meta_graph
sample_node = 1
if has_prop(meta_graph_ref, sample_node, :contact_restricting)
    contact_restricting = get_prop(meta_graph_ref, sample_node, :contact_restricting)
    println("Node $sample_node contact restricting behavior: $contact_restricting")
end

=== Example: Accessing Node Attributes ===
Agent 1 (S, high risk):
  Age: 76.0, Gender: MALE, Protection Score: 2.0
Agent 2 (S, low risk):
  Age: 62.0, Gender: MALE, Protection Score: 2.0
Agent 3 (S, high risk):
  Age: 79.0, Gender: MALE, Protection Score: 4.0

Direct MetaGraph access example:
Node 1 contact restricting behavior: true


### Benefits of Using Node Attributes

By integrating the JSON metadata into a MetaGraph, we now have access to rich node attributes that can enhance our epidemic modeling:

1. **Age-based risk stratification**: Older individuals (>65) are automatically assigned high risk
2. **Behavioral factors**: We can incorporate `contact_restricting` and `protective_measure_score` into transmission dynamics
3. **Perceived severity**: The `perceived_severity_high` attribute can influence agent behavior
4. **Demographic analysis**: Gender and age distributions provide realistic population structure

### Potential Extensions

The node attributes can be used to:
- **Modify transmission probabilities** based on protective measure scores
- **Implement contact reduction** for agents with `contact_restricting = true`
- **Create age-stratified hospitalization rates**
- **Model risk perception effects** on behavior
- **Analyze demographic-specific outcomes**

This approach demonstrates how real-world network data with metadata can create more realistic and nuanced epidemic models compared to synthetic networks alone.

## Model Definition

Now we'll set up the epidemic model using the same modular approach as in the other tutorials. The model components are defined in separate source files for modularity and reusability.

In [ ]:
include("src/create_graph.jl")

# Agent creation: agents of type Person and properties status, days_infected and risk
@agent struct Person(GraphAgent)
    status::Symbol = :S #((S)usceptible, (I)nfected, (H)ospitalized, (R)ecovered)
    days_infected::Int = 0 # number of days since infection
    risk::Symbol = :high # something to differentiate agents (here, high and low risk)
end

include("src/initialize.jl")
include("src/agent_step.jl")

# Model step: keep track of the infection numbers
function model_step!(model::ABM)
    model.susceptible_count = sum([model[i].status == :S for i in 1:nv(model.graph)])
    model.infected_count = sum([model[i].status == :I for i in 1:nv(model.graph)])
    model.hospitalized_count = sum([model[i].status == :H for i in 1:nv(model.graph)])
    model.recovered_count = sum([model[i].status == :R for i in 1:nv(model.graph)])
end

## Model Initialization with Edgelist Network

There are two ways to initialize the model with the edgelist network:

### Method 1: Using the `custom_graph` parameter
Pass the pre-loaded graph directly to the initialization function.

### Method 2: Using the `:edgelist` network type
Let the initialization function load the edgelist automatically.

We'll demonstrate both approaches:

In [ ]:
# Method 1: Use the custom_graph parameter to pass the pre-loaded graph
model = initialize_old(
    network_type = :custom,  # Will be set automatically when using custom_graph
    custom_graph = graph,
    patient_zero = :random,
    high_risk = :random,
    fraction_high_risk = 0.1,
    trans_prob = 0.1,
    days_to_recovered = 14,
    seed = 42,
    low_risk_factor = 1.0
)

println("Model initialized with $(model.n_nodes) agents")
println("Network type: $(model.network_type)")
println("Mean degree: $(model.mean_degree)")

In [ ]:
# Alternative Method 2: Use the edgelist network type to load directly
# Uncomment the code below to try this approach instead:

# model_alt = initialize_old(
#     network_type = :edgelist,
#     edgelist_path = "degs/network",
#     patient_zero = :random,
#     high_risk = :random,
#     fraction_high_risk = 0.1,
#     trans_prob = 0.1,
#     days_to_recovered = 14,
#     seed = 42,
#     low_risk_factor = 1.0
# )

# println("Alternative model initialized with $(model_alt.n_nodes) agents")
# println("Network type: $(model_alt.network_type)")
# println("Mean degree: $(model_alt.mean_degree)")

### Why the initialize function didn't work initially

The issue was that the current `initialize` function doesn't support the `custom_graph` parameter that we were trying to use. The codebase has two versions of the initialize function:

1. **`initialize_old`** - supports `custom_graph` and other advanced parameters
2. **`initialize`** - newer version but with fewer parameters

We used `initialize_old` above to fix the issue. Alternatively, we could also copy our processed edgelist file to match the default path expected by the `:edgelist` network type.

## Network Analysis

Now we can analyze the network structure using the regular workflow functions. This includes calculating various network metrics and centrality measures.

In [ ]:
include("src/analyze_graph.jl")
include("src/plotting.jl")

# Analyze the graph
println("Analyzing the edgelist network...")
graph_analysis = analyze_graph(model.graph)

display(graph_analysis["summary"])

display(first(graph_analysis["centrality"], 10))

In [ ]:
# Save the analysis to CSV files
CSV.write("data/graph_summary_$(model.network_type)_mdeg_$(model.mean_degree).csv", graph_analysis["summary"])
CSV.write("data/centrality_$(model.network_type)_mdeg_$(model.mean_degree).csv", graph_analysis["centrality"])

println("Analysis results saved to data/ directory")

## Epidemic Simulation

Let's run an epidemic simulation on the edgelist network and visualize the dynamics.

In [ ]:
# Run a single simulation to test dynamics
println("Running SIHR epidemic simulation...")
adata = [:status]
mdata = [:susceptible_count, :infected_count, :hospitalized_count, :recovered_count]
adf, mdf = run!(model, 100; adata, mdata)

println("Simulation completed!")
println("Final outbreak size: $(mdf[end, :recovered_count]) out of $(model.n_nodes) agents")
println("Peak hospitalized: $(maximum(mdf.hospitalized_count)) agents")
println("Attack rate: $(round(100 * mdf[end, :recovered_count] / model.n_nodes, digits=1))%")

In [ ]:
# Plot the epidemic dynamics
dynamics_plot = plot_epidemic_trajectories(mdf, model.network_type)
display(dynamics_plot)
savefig(dynamics_plot, "figures/dynamics_$(model.network_type)_mdeg_$(model.mean_degree).pdf")

println("Dynamics plot saved to figures/ directory")

## Network Comparison

One of the advantages of integrating edgelist networks into the regular workflow is the ability to compare them with synthetic network types. Let's compare our edgelist network with random and small-world networks.

In [ ]:
# Compare network metrics across different network types
# Use the same number of nodes as the edgelist network for fair comparison
include("src/plotting.jl")

comparison_plot = plot_network_metrics_comparison(
    network_types=[:random, :smallworld, :edgelist], 
    mean_degree=model.mean_degree, 
    n_nodes=model.n_nodes  # Use actual edgelist network size
)

display(comparison_plot)
println("Network comparison completed with $(model.n_nodes) nodes for all network types!")

In [ ]:
# Also generate centrality comparison for the same network types
println("\nGenerating centrality comparison for network structure analysis...")
centrality_comparison = plot_centrality_comparison(
    network_types = [:random, :smallworld, :preferential, :edgelist],
    mean_degree = model.mean_degree,
    n_nodes = model.n_nodes,
    link_axes = true
)

display(centrality_comparison)
println("Centrality comparison completed! Results saved to figures/ directory.")

## Epidemic Comparison

Now let's run multiple epidemic simulations to compare the dynamics across different network types. This will show how the network structure affects epidemic spread patterns.

In [ ]:
# Load the run_simulations function needed for epidemic comparison
include("src/run_simulations.jl")

# Run epidemic comparison across network types
println("Running epidemic comparison between network types...")
println("This may take a few minutes as it runs multiple simulations...")

# Note: Proportionate mixing networks may have issues with small network sizes
epidemic_comparison = run_and_plot_comparison(
    network_types = [:random, :smallworld, :preferential, :edgelist],
    mean_degree = model.mean_degree,
    n_nodes = model.n_nodes,
    trans_prob = 0.05,
    n_steps = 100
)

display(epidemic_comparison)
println("Epidemic comparison completed! Results saved to figures/ and output/ directories.")